In [3]:
import numpy as np
import pandas as pd
import plotly.express as px
from scipy import stats

print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
pd.set_option("display.width", 80)

NumPy: 2.5.3
Pandas: 3.0.6


### Load the Data Sets

In [4]:
df_fixtures = pd.read_csv("wc_2026_fixtures.csv")
df_teams = pd.read_csv("wc_2026_teams.csv")
df_editions = pd.read_csv("wc_all_editions.csv")
df_matches = pd.read_csv("wc_all_matches.csv")
df_scorers = pd.read_csv("wc_top_scorers.csv")

**Finding the missing values (each DataFrame sepretly)**

### First, Fixtures data set

**Data Frame info**

In [ ]:
print(f"\nShape:\n{df_fixtures.shape}")
print(f"\nColumns:\n{df_fixtures.columns.tolist()}")
print(f"\nData types:\n{df_fixtures.dtypes}")
print("\nFirst 5 rows:")
display(df_fixtures)
print("\nInfo:")
df_fixtures.info()
print("dataframe descirebtion: ")
display(df_fixtures.describe().T)
df_fixtures.describe(include="str").T

In [4]:
df_fixtures.isnull().sum()
# missing values are knockout matchess

group                  32
stage                   0
team1                   0
team2                   0
venue                   0
city                    0
country                 0
date                    0
kickoff_et              0
team1_confederation    32
team1_fifa_rank        32
team1_coach            32
team2_confederation    32
team2_fifa_rank        32
team2_coach            32
dtype: int64

**Finding the duplicated values (each DataFrame sepretly)**

In [5]:
df_fixtures.duplicated().head()
# no exact duplicates!

0    False
1    False
2    False
3    False
4    False
dtype: bool

In [20]:
grp = df_fixtures[df_fixtures["stage"] == "Group Stage"].copy()
grp["pair"] = grp.apply(lambda r: frozenset([r["team1"], r["team2"]]), axis=1)
grp[grp.duplicated(["group", "pair"], keep=False)][["group","team1","team2","date","kickoff_et", "venue"]]

# the issue is in group c, there should 6 distict matches, yet there is 4 distict and 2 logical dupes, 
# soo we need to delete the wrong ones (rows 14 and 15)
# and add the missing matches which are Brazil – Haiti and Scotland – Morocco


,group,team1,team2,date,kickoff_et,venue
14,C,Brazil,Scotland,2026-06-19,21:00 ET,MetLife Stadium
15,C,Morocco,Haiti,2026-06-19,21:00 ET,MetLife Stadium
16,C,Scotland,Brazil,2026-06-24,21:00 ET,Gillette Stadium
17,C,Morocco,Haiti,2026-06-24,21:00 ET,Gillette Stadium


In [21]:
fx = df_fixtures.copy()

assert fx.loc[14, ["team1", "team2"]].tolist() == ["Brazil", "Scotland"]
assert fx.loc[15, ["team1", "team2"]].tolist() == ["Morocco", "Haiti"]

fixes = {
    14: {"team1": "Brazil", "team2": "Haiti", "venue": "Lincoln Financial Field",
            "city": "Philadelphia", "kickoff_et": "21:00 ET"},
    15: {"team1": "Scotland", "team2": "Morocco", "venue": "Gillette Stadium",
            "city": "Foxborough", "kickoff_et": "18:00 ET"},
    16: {"venue": "Hard Rock Stadium", "city": "Miami", "kickoff_et": "18:00 ET"},
    17: {"venue": "Mercedes-Benz Stadium", "city": "Atlanta", "kickoff_et": "18:00 ET"},
}
fx["manually_corrected"] = False                      # audit trail
for idx, vals in fixes.items():
    for col, v in vals.items():
        fx.loc[idx, col] = v
    fx.loc[idx, "manually_corrected"] = True

# Rank/confederation/coach must follow the new team names: rebuild from df_teams
# df_teams is a lookup table and we get each team's information from it
t = df_teams.set_index("team")
m = fx["stage"].eq("Group Stage")
for side in ("team1", "team2"):
    fx.loc[m, f"{side}_confederation"] = fx.loc[m, side].map(t["confederation"])
    fx.loc[m, f"{side}_fifa_rank"]     = fx.loc[m, side].map(t["fifa_rank"])
    fx.loc[m, f"{side}_coach"]         = fx.loc[m, side].map(t["coach"])

In [ ]:
g = fx[m].copy()
g["pair"] = g.apply(lambda r: frozenset([r["team1"], r["team2"]]), axis=1)

assert (g.groupby("group")["pair"].nunique() == 6).all()          # 6 distinct pairings per group
teams_played = pd.concat([g["team1"], g["team2"]]).value_counts()
assert (teams_played == 3).all() and len(teams_played) == 48      # each team plays 3, 48 teams
assert g[["team1_fifa_rank", "team2_fifa_rank"]].notna().all().all() # check that all teams have fifa rank, confederation and coach after the fix

In [ ]:
print(fx[fx.duplicated(["venue", "date", "kickoff_et"], keep=False)]
    .sort_values(["date", "venue"])[["group", "team1", "team2", "date", "kickoff_et", "venue"]])
# now the problem is that there are 2 matches with the same venue, date and kickoff time, which is not possible.

   group         team1         team2        date kickoff_et              venue
26     E       Germany   Ivory Coast  2026-06-20   19:00 ET        NRG Stadium
32     F   Netherlands        Sweden  2026-06-20   19:00 ET        NRG Stadium
4      A       Czechia        Mexico  2026-06-24   21:00 ET     Estadio Azteca
5      A  South Africa   South Korea  2026-06-24   21:00 ET     Estadio Azteca
22     D       Türkiye           USA  2026-06-25   21:00 ET       AT&T Stadium
23     D      Paraguay     Australia  2026-06-25   21:00 ET       AT&T Stadium
28     E       Ecuador       Germany  2026-06-25   18:00 ET       AT&T Stadium
29     E       Curaçao   Ivory Coast  2026-06-25   18:00 ET       AT&T Stadium
34     F         Japan        Sweden  2026-06-26   15:00 ET       AT&T Stadium
35     F       Tunisia   Netherlands  2026-06-26   15:00 ET       AT&T Stadium
46     H    Cape Verde  Saudi Arabia  2026-06-26   20:00 ET      Estadio Akron
47     H       Uruguay         Spain  2026-06-26   2

In [25]:
def pair_key(a, b):
    # Order-independent text key: "Brazil | Haiti" for either order
    return " | ".join(sorted([a, b]))

fx["pair_key"] = [pair_key(a, b) for a, b in zip(fx["team1"], fx["team2"])]

In [ ]:
# correct kickoff times for the duplicate matches (same venue, date, kickoff time) so that they are unique
# sources: 
# https://www.fifa.com/en/tournaments/mens/worldcup/canadamexicousa2026/scores-fixtures
# https://www.fifa.com/tickets 
# https://en.wikipedia.org/wiki/2026_FIFA_World_Cup
patch = pd.DataFrame(
    [
        # team1, team2, venue, city, country, kickoff_et
        (
            "Germany",
            "Ivory Coast",
            "BMO Field",
            "Toronto",
            "Canada",
            "16:00 ET",
        ),
        (
            "Netherlands",
            "Sweden",
            "NRG Stadium",
            "Houston",
            "USA",
            "17:00 ET",
        ),
        (
            "Czechia",
            "Mexico",
            "Estadio Azteca",
            "Mexico City",
            "Mexico",
            "21:00 ET",
        ),
        (
            "South Africa",
            "South Korea",
            "Estadio BBVA",
            "Monterrey",
            "Mexico",
            "21:00 ET",
        ),
        (
            "Türkiye",
            "USA",
            "SoFi Stadium",
            "Los Angeles",
            "USA",
            "22:00 ET",
        ),
        (
            "Paraguay",
            "Australia",
            "Levi's Stadium",
            "San Francisco",
            "USA",
            "22:00 ET",
        ),
        (
            "Ecuador",
            "Germany",
            "MetLife Stadium",
            "New York / New Jersey",
            "USA",
            "16:00 ET",
        ),
        (
            "Curaçao",
            "Ivory Coast",
            "Lincoln Financial Field",
            "Philadelphia",
            "USA",
            "16:00 ET",
        ),
        (
            "Japan",
            "Sweden",
            "AT&T Stadium",
            "Dallas",
            "USA",
            "19:00 ET",
        ),
        (
            "Tunisia",
            "Netherlands",
            "Arrowhead Stadium",
            "Kansas City",
            "USA",
            "19:00 ET",
        ),
        (
            "Cape Verde",
            "Saudi Arabia",
            "NRG Stadium",
            "Houston",
            "USA",
            "20:00 ET",
        ),
        (
            "Uruguay",
            "Spain",
            "Estadio Akron",
            "Guadalajara",
            "Mexico",
            "20:00 ET",
        ),
        (
            "Norway",
            "France",
            "Gillette Stadium",
            "Boston",
            "USA",
            "15:00 ET",
        ),
        (
            "Senegal",
            "Iraq",
            "BMO Field",
            "Toronto",
            "Canada",
            "15:00 ET",
        ),
        (
            "Colombia",
            "Portugal",
            "Hard Rock Stadium",
            "Miami",
            "USA",
            "19:30 ET",
        ),
        (
            "DR Congo",
            "Uzbekistan",
            "Mercedes-Benz Stadium",
            "Atlanta",
            "USA",
            "19:30 ET",
        ),
        (
            "Panama",
            "England",
            "MetLife Stadium",
            "New York / New Jersey",
            "USA",
            "17:00 ET",
        ),
        (
            "Croatia",
            "Ghana",
            "Lincoln Financial Field",
            "Philadelphia",
            "USA",
            "17:00 ET",
        ),
    ],
    columns=["team1", "team2", "venue", "city", "country", "kickoff_et"],
)
patch

,team1,team2,venue,city,country,kickoff_et
0,Germany,Ivory Coast,BMO Field,Toronto,Canada,16:00 ET
1,Netherlands,Sweden,NRG Stadium,Houston,USA,17:00 ET
2,Czechia,Mexico,Estadio Azteca,Mexico City,Mexico,21:00 ET
3,South Africa,South Korea,Estadio BBVA,Monterrey,Mexico,21:00 ET
4,Türkiye,USA,SoFi Stadium,Los Angeles,USA,22:00 ET
5,Paraguay,Australia,Levi's Stadium,San Francisco,USA,22:00 ET
6,Ecuador,Germany,MetLife Stadium,New York / New Jersey,USA,16:00 ET
7,Curaçao,Ivory Coast,Lincoln Financial Field,Philadelphia,USA,16:00 ET
8,Japan,Sweden,AT&T Stadium,Dallas,USA,19:00 ET
9,Tunisia,Netherlands,Arrowhead Stadium,Kansas City,USA,19:00 ET


In [54]:
patch["pair_key"] = [pair_key(a, b) for a, b in zip(patch["team1"], patch["team2"])]

if "manually_corrected" not in fx.columns:
    fx["manually_corrected"] = False

for row in patch.itertuples(index=False):             # one pass per correction
    hit = fx["pair_key"].eq(row.pair_key)              # True on the matching fixture row
    assert hit.sum() == 1, row.pair_key                # must match exactly one row, else stop
    for col in ["venue", "city", "country", "kickoff_et"]:
        fx.loc[hit, col] = getattr(row, col)           # overwrite that cell
    fx.loc[hit, "manually_corrected"] = True


fx[fx.duplicated(["venue", "date", "kickoff_et"], keep=False)].sort_values(["date", "venue"])[["group", "team1", "team2", "date", "kickoff_et", "venue"]]

,group,team1,team2,date,kickoff_et,venue


***Small corrections***

In [75]:
venues = {
    "AT&T Stadium": "Arlington",
    "Gillette Stadium": "Foxborough",
    "MetLife Stadium": "East Rutherford",
    "SoFi Stadium": "Inglewood",
}
fx["city"] = fx["venue"].map(venues).fillna(fx["city"]) 
# Correct the physical city for these four stadiums; leave all other cities unchanged

fx.loc[75, ["venue", "city", "country"]] = ["Estadio BBVA", "Monterrey", "Mexico"]
fx.loc[90, ["venue", "city", "country"]] = ["NRG Stadium", "Houston", "USA"]
fx.loc[98, ["venue", "city", "country"]] = ["SoFi Stadium", "Inglewood", "USA"]
# these 3 stadiums were misnamed (Rose Bowl) in the original data, so we correct them here
list(fx.columns)

['group',
 'stage',
 'team1',
 'team2',
 'venue',
 'city',
 'country',
 'date',
 'kickoff_et',
 'team1_confederation',
 'team1_fifa_rank',
 'team1_coach',
 'team2_confederation',
 'team2_fifa_rank',
 'team2_coach',
 'manually_corrected',
 'pair_key']

****Validation****

In [74]:
g = fx[fx["stage"].eq("Group Stage")]
assert not g.duplicated(["venue", "date", "kickoff_et"]).any()          # no same-time clashes
assert fx.groupby("venue")[["city", "country"]].nunique().eq(1).all().all()  # each venue = one city/country
assert fx["venue"].nunique() == 16                                    # 16 host stadiums
print(fx.groupby("city").size().sort_values(ascending=False))   # matches per city, 16 cities

city
Arlington          11
East Rutherford    11
Inglewood           8
Houston             8
Foxborough          7
Toronto             7
Philadelphia        7
Miami               7
Kansas City         6
Guadalajara         6
Atlanta             6
Seattle             6
Monterrey           5
Mexico City         4
Vancouver           4
San Francisco       1
dtype: int64


**this block of code was used to identify dupes within venues names and their location**

In [66]:
n = fx.groupby("venue")[["city", "country"]].nunique()
bad = n[n.gt(1).any(axis=1)]            # venues with more than 1 distinct city or country
print(bad)

# Show the actual conflicting spellings and how many rows use each
print(fx[fx["venue"].isin(bad.index)]
        .groupby(["venue", "city", "country"]).size())

Empty DataFrame
Columns: [city, country]
Index: []
Series([], dtype: int64)


**Fix the "str" type for the date feature**

In [68]:
fx["date"] = pd.to_datetime(fx["date"], format="%Y-%m-%d")
assert pd.api.types.is_datetime64_any_dtype(fx["date"])
print(fx["date"].dtype, fx["date"].min().date(), fx["date"].max().date())

datetime64[us] 2026-06-11 2026-07-19


**Save the Corrected DataFrame**

In [70]:
out = fx.drop(columns="pair_key")      # helper column, not part of the dataset
out.to_csv("wc_2026_fixtures_clean.csv", index=False, date_format="%Y-%m-%d")

### Second, the Teams Dataset

**Data Frame info**

In [ ]:
print(f"\nShape:\n{df_teams.shape}")
print(f"\nColumns:\n{df_teams.columns.tolist()}")
print(f"\nData types:\n{df_teams.dtypes}")
print("\nAll rows:")
display(df_teams)
print("\nInfo:")
df_teams.info()
print("dataframe descirebtion: ")
display(df_teams.describe().T)
df_teams.describe(include="str").T
# no need to check for duplicates in the teams dataset, as each team is unique and has a unique FIFA code.

**Check for Nulls and Dupes**

In [ ]:
df_teams.isnull().sum()
# no missing values in the teams dataset, as each team is unique and has a unique FIFA code.

team              0
group             0
confederation     0
fifa_rank         0
coach             0
best_wc_result    0
debut_2026        0
dtype: int64

In [ ]:
df_teams.duplicated().sum()
# no duplicate rows in the teams dataset, as each team is unique and has a unique FIFA code.

np.int64(0)

In [92]:
assert df_teams["team"].nunique() == 48
assert df_teams["group"].nunique() == 12
assert df_teams.groupby("group").size().eq(4).all()
# each group has 4 teams, and there are 12 groups, so 48 teams in total.

### Third, Editions Datasets

**Data Frame info**

In [5]:
print(f"\nShape:\n{df_editions.shape}")
print(f"\nColumns:\n{df_editions.columns.tolist()}")
print(f"\nData types:\n{df_editions.dtypes}")
print("\nAll rows:")
display(df_editions)
print("\nInfo:")
df_editions.info()
print("dataframe descirebtion: ")
display(df_editions.describe().T)
df_editions.describe(include="str").T


Shape:
(22, 20)

Columns:
['edition', 'year', 'host', 'champion', 'runner_up', 'third_place', 'fourth_place', 'teams', 'matches', 'goals', 'goals_per_match', 'attendance', 'top_scorer', 'top_scorer_country', 'top_scorer_goals', 'start_date', 'end_date', 'final_city', 'host_won', 'format']

Data types:
edition                 int64
year                    int64
host                      str
champion                  str
runner_up                 str
third_place               str
fourth_place              str
teams                   int64
matches                 int64
goals                   int64
goals_per_match       float64
attendance              int64
top_scorer                str
top_scorer_country        str
top_scorer_goals        int64
start_date                str
end_date                  str
final_city                str
host_won                  str
format                    str
dtype: object

All rows:


,edition,year,host,champion,runner_up,third_place,fourth_place,teams,matches,goals,goals_per_match,attendance,top_scorer,top_scorer_country,top_scorer_goals,start_date,end_date,final_city,host_won,format
0,1,1930,Uruguay,Uruguay,Argentina,United States,Yugoslavia,13,18,70,3.89,590549,Guillermo Stábile,Argentina,8,July 13,July 30,Montevideo,No,Round Robin
1,2,1934,Italy,Italy,Czechoslovakia,Germany,Austria,16,17,70,4.12,363000,Oldřich Nejedlý,Czechoslovakia,5,May 27,June 10,Rome,Yes,Knockout
2,3,1938,France,Italy,Hungary,Brazil,Sweden,15,18,84,4.67,375700,Leônidas,Brazil,7,June 4,June 19,Paris,Yes,Knockout
3,4,1950,Brazil,Uruguay,Brazil,Sweden,Spain,13,22,88,4.00,1045246,Ademir,Brazil,8,June 24,July 16,Rio de Janeiro,No,Round Robin Final
4,5,1954,Switzerland,West Germany,Hungary,Austria,Uruguay,16,26,140,5.38,768607,Sándor Kocsis,Hungary,11,June 16,July 4,Bern,No,Knockout
5,6,1958,Sweden,Brazil,Sweden,France,West Germany,16,35,126,3.60,819810,Just Fontaine,France,13,June 8,June 29,Stockholm,No,Knockout
6,7,1962,Chile,Brazil,Czechoslovakia,Chile,Yugoslavia,16,32,89,2.78,893172,Florian Albert / Garrincha / Valentin Ivanov /...,Multiple,4,May 30,June 17,Santiago,No,Knockout
7,8,1966,England,England,West Germany,Portugal,Soviet Union,16,32,89,2.78,1563135,Eusébio,Portugal,9,July 11,July 30,London,Yes,Knockout
8,9,1970,Mexico,Brazil,Italy,West Germany,Uruguay,16,32,95,2.97,1603975,Gerd Müller,West Germany,10,May 31,June 21,Mexico City,No,Knockout
9,10,1974,West Germany,West Germany,Netherlands,Poland,Brazil,16,38,97,2.55,1865753,Grzegorz Lato,Poland,7,June 13,July 7,Munich,Yes,Knockout



Info:
<class 'pandas.DataFrame'>
RangeIndex: 22 entries, 0 to 21
Data columns (total 20 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   edition             22 non-null     int64  
 1   year                22 non-null     int64  
 2   host                22 non-null     str    
 3   champion            22 non-null     str    
 4   runner_up           22 non-null     str    
 5   third_place         22 non-null     str    
 6   fourth_place        22 non-null     str    
 7   teams               22 non-null     int64  
 8   matches             22 non-null     int64  
 9   goals               22 non-null     int64  
 10  goals_per_match     22 non-null     float64
 11  attendance          22 non-null     int64  
 12  top_scorer          22 non-null     str    
 13  top_scorer_country  22 non-null     str    
 14  top_scorer_goals    22 non-null     int64  
 15  start_date          22 non-null     str    
 16  end_date      

,count,mean,std,min,25%,50%,75%,max
edition,22.0,1.150000e+01,6.493587e+00,1.00,6.2500,11.5,1.675000e+01,22.00
year,22.0,1.978909e+03,2.773842e+01,1930.00,1959.0000,1980.0,2.001000e+03,2022.00
teams,22.0,2.222727e+01,7.602830e+00,13.00,16.0000,20.0,3.200000e+01,32.00
matches,22.0,4.381818e+01,1.763286e+01,17.00,32.0000,45.0,6.400000e+01,64.00
goals,22.0,1.236364e+02,3.484188e+01,70.00,90.5000,129.0,1.467500e+02,172.00
goals_per_match,22.0,3.065909e+00,8.400548e-01,2.21,2.5725,2.7,3.442500e+00,5.38
attendance,22.0,1.981513e+06,1.098572e+06,363000.00,931190.5000,1816015.0,2.970101e+06,3587538.00
top_scorer_goals,22.0,7.090909e+00,2.158242e+00,4.00,6.0000,6.0,8.000000e+00,13.00


,count,unique,top,freq
host,22,18,Italy,2
champion,22,9,Brazil,5
runner_up,22,11,Argentina,3
third_place,22,15,Germany,3
fourth_place,22,17,Uruguay,3
top_scorer,22,22,Guillermo Stábile,1
top_scorer_country,22,15,Brazil,3
start_date,22,18,May 31,3
end_date,22,19,July 30,2
final_city,22,18,Rome,2


In [11]:
fe = df_editions.copy()   # copy, so re-running the cell never double-converts the original

MONTHS = {m: i for i, m in enumerate(
    ["January","February","March","April","May","June",
    "July","August","September","October","November","December"], start=1)}

def parse_edition_date(year: pd.Series, text: pd.Series) -> pd.Series:
    # "July 13" (maybe with \xa0 / extra spaces) -> split into month name + day
    parts = (text.str.replace(r"\s+", " ", regex=True)   # \s also matches \xa0 in Python 3
                .str.strip()
                .str.split(" ", n=1, expand=True))
    month = parts[0].map(MONTHS)
    day = parts[1].astype(int)
    assert month.notna().all(), parts[month.isna()]       # fail loudly on unknown month names
    return pd.to_datetime({"year": year, "month": month, "day": day})

fe["start_date"] = parse_edition_date(fe["year"], fe["start_date"])
fe["end_date"]   = parse_edition_date(fe["year"], fe["end_date"])

assert (fe["end_date"] >= fe["start_date"]).all()
fe[["start_date", "end_date", "year"]]

,start_date,end_date,year
0,1930-07-13,1930-07-30,1930
1,1934-05-27,1934-06-10,1934
2,1938-06-04,1938-06-19,1938
3,1950-06-24,1950-07-16,1950
4,1954-06-16,1954-07-04,1954
5,1958-06-08,1958-06-29,1958
6,1962-05-30,1962-06-17,1962
7,1966-07-11,1966-07-30,1966
8,1970-05-31,1970-06-21,1970
9,1974-06-13,1974-07-07,1974


In [12]:
fe["format"].unique()

<StringArray>
['Round Robin', 'Knockout', 'Round Robin Final']
Length: 3, dtype: str

**Corrected Formats**

In [13]:
FORMAT_MAP = {
    1930: "Groups + Knockout",
    1934: "Knockout",
    1938: "Knockout",
    1950: "Groups + Final Round",
    **{y: "Groups + Knockout" for y in
        (1954, 1958, 1962, 1966, 1970, 1986, 1990, 1994, 1998, 2002, 2006, 2010, 2014, 2018, 2022)},
    1974: "Groups + Second Group Stage",
    1978: "Groups + Second Group Stage",
    1982: "Groups + Second Group Stage",
}

fe["format_raw"] = fe["format"]                       # keep original for the audit trail
fe["format"] = fe["year"].map(FORMAT_MAP)

assert fe["format"].notna().all()                     # every year covered
print(fe.groupby("format")["year"].apply(list))
fe[["format", "format_raw"]]

format
Groups + Final Round                                                      [1950]
Groups + Knockout              [1930, 1954, 1958, 1962, 1966, 1970, 1986, 199...
Groups + Second Group Stage                                   [1974, 1978, 1982]
Knockout                                                            [1934, 1938]
Name: year, dtype: object


,format,format_raw
0,Groups + Knockout,Round Robin
1,Knockout,Knockout
2,Knockout,Knockout
3,Groups + Final Round,Round Robin Final
4,Groups + Knockout,Knockout
5,Groups + Knockout,Knockout
6,Groups + Knockout,Knockout
7,Groups + Knockout,Knockout
8,Groups + Knockout,Knockout
9,Groups + Second Group Stage,Knockout


In [21]:
# in 1930 and 1938, the host won the title, but the "host_won" column says "No", so we correct that here
should_be = fe["host"] == fe["champion"]                 # True when the host won the title
bad = fe[fe["host_won"].eq("Yes") != should_be]
print(bad[["year", "host", "champion", "host_won"]])     # expect exactly 1930 and 1938

fe["host_won_raw"] = fe["host_won"]                      # keep the original for the audit trail
fe["host_won"] = should_be    

    year          host      champion  host_won
0   1930       Uruguay       Uruguay      True
1   1934         Italy         Italy      True
7   1966       England       England      True
9   1974  West Germany  West Germany      True
10  1978     Argentina     Argentina      True
15  1998        France        France      True


In [22]:
CANON = {"West Germany": "Germany", "United States": "USA",
            "Turkey": "Türkiye", "Czechoslovakia": "Czechia"}

for col in ["champion", "runner_up", "third_place", "fourth_place", "top_scorer_country", "host"]:
    fe[f"{col}_canon"] = fe[col].replace(CANON)

print(fe[["year", "champion", "champion_canon", "runner_up", "runner_up_canon", "host", "host_canon"]])


    year      champion champion_canon       runner_up runner_up_canon  \
0   1930       Uruguay        Uruguay       Argentina       Argentina   
1   1934         Italy          Italy  Czechoslovakia         Czechia   
2   1938         Italy          Italy         Hungary         Hungary   
3   1950       Uruguay        Uruguay          Brazil          Brazil   
4   1954  West Germany        Germany         Hungary         Hungary   
5   1958        Brazil         Brazil          Sweden          Sweden   
6   1962        Brazil         Brazil  Czechoslovakia         Czechia   
7   1966       England        England    West Germany         Germany   
8   1970        Brazil         Brazil           Italy           Italy   
9   1974  West Germany        Germany     Netherlands     Netherlands   
10  1978     Argentina      Argentina     Netherlands     Netherlands   
11  1982         Italy          Italy    West Germany         Germany   
12  1986     Argentina      Argentina    West Germa

In [28]:
assert fe["year"].nunique() == 22
assert not fe.duplicated().any()
assert fe["year"].is_unique and fe["edition"].is_unique
assert (fe["start_date"].dt.year == fe["year"]).all()
assert (fe["end_date"] >= fe["start_date"]).all()
assert np.isclose(fe["goals"] / fe["matches"], fe["goals_per_match"], atol=0.005).all()

In [ ]:
dm = df_matches.copy()
for c in ("team1", "team2"):
    dm[f"{c}_canon"] = dm[c].replace(CANON)         

# Check A: the two finalists in the matches file must be {champion, runner_up}
finals = dm[dm["stage"].eq("Final")].set_index("year")
finalists = finals.apply(lambda r: {r["team1_canon"], r["team2_canon"]}, axis=1)
expected  = fe.set_index("year").apply(lambda r: {r["champion_canon"], r["runner_up_canon"]}, axis=1)

cmp = pd.DataFrame({"matches_file": finalists, "editions_file": expected}).dropna()  # 1950 has no "Final" row
mismatch = cmp[cmp.apply(lambda r: r["matches_file"] != r["editions_file"], axis=1)]
print(mismatch)            # an empty result means the files agree

# Check B: top scorer's goals must agree with the scorers file
chk = fe.set_index("year")[["top_scorer_goals"]].join(df_scorers.set_index("year")["goals"])
print(chk[chk["top_scorer_goals"] != chk["goals"]])      # empty result means they agree

Empty DataFrame
Columns: [matches_file, editions_file]
Index: []
Empty DataFrame
Columns: [top_scorer_goals, goals]
Index: []


In [54]:
names_editions = set(pd.concat([fe[c] for c in
                ["champion", "runner_up", "third_place", "fourth_place"]]).unique())
names_matches  = set(pd.concat([df_matches["team1"], df_matches["team2"]]).unique())

print(sorted(names_editions - names_matches))   # only in editions
print(sorted(names_matches - names_editions))   # only in matches

['United States']
['Algeria', 'Australia', 'Bolivia', 'Cameroon', 'Colombia', 'Costa Rica', 'Denmark', 'East Germany', 'Ecuador', 'Ghana', 'Ireland', 'Japan', 'Mexico', 'Nigeria', 'North Korea', 'Northern Ireland', 'Norway', 'Paraguay', 'Peru', 'Qatar', 'Romania', 'Russia', 'Saudi Arabia', 'Scotland', 'Senegal', 'South Africa', 'Switzerland', 'USA', 'Ukraine']


In [55]:
fe.to_csv("wc_all_editions_clean.csv", index=False, date_format="%Y-%m-%d")

### Forth, Matches Dataset

**Date Frame Info**

In [30]:
print(f"\nShape:\n{df_matches.shape}")
print(f"\nColumns:\n{df_matches.columns.tolist()}")
print(f"\nData types:\n{df_matches.dtypes}")
print("\nAll rows:")
display(df_matches)
print("\nInfo:")
df_matches.info()
print("dataframe descirebtion: ")
display(df_matches.describe().T)
df_matches.describe(include="str").T


Shape:
(184, 11)

Columns:
['year', 'stage', 'team1', 'score1', 'score2', 'team2', 'venue', 'city', 'country', 'date', 'notes']

Data types:
year       int64
stage        str
team1        str
score1     int64
score2     int64
team2        str
venue        str
city         str
country      str
date         str
notes        str
dtype: object

All rows:


,year,stage,team1,score1,score2,team2,venue,city,country,date,notes
0,1930,Group Stage,France,4,1,Mexico,Estadio Centenario,Montevideo,Uruguay,1930-07-13,First ever WC match
1,1930,Group Stage,USA,3,0,Belgium,Estadio Centenario,Montevideo,Uruguay,1930-07-13,NaN
2,1930,Group Stage,Yugoslavia,2,1,Brazil,Estadio Centenario,Montevideo,Uruguay,1930-07-14,NaN
3,1930,Group Stage,Uruguay,1,0,Peru,Estadio Centenario,Montevideo,Uruguay,1930-07-18,NaN
4,1930,Group Stage,Argentina,6,3,USA,Estadio Centenario,Montevideo,Uruguay,1930-07-26,NaN
...,...,...,...,...,...,...,...,...,...,...,...
179,2022,Quarter-final,France,2,1,England,Al Bayt Stadium,Al Khor,Qatar,2022-12-10,Kane pen miss
180,2022,Semi-final,Argentina,3,0,Croatia,Lusail Stadium,Lusail,Qatar,2022-12-13,Messi pen + Álvarez 2
181,2022,Semi-final,France,2,0,Morocco,Al Bayt Stadium,Al Khor,Qatar,2022-12-14,NaN
182,2022,3rd Place,Croatia,2,1,Morocco,Khalifa International,Al Rayyan,Qatar,2022-12-17,NaN



Info:
<class 'pandas.DataFrame'>
RangeIndex: 184 entries, 0 to 183
Data columns (total 11 columns):
 #   Column   Non-Null Count  Dtype
---  ------   --------------  -----
 0   year     184 non-null    int64
 1   stage    184 non-null    str  
 2   team1    184 non-null    str  
 3   score1   184 non-null    int64
 4   score2   184 non-null    int64
 5   team2    184 non-null    str  
 6   venue    184 non-null    str  
 7   city     184 non-null    str  
 8   country  184 non-null    str  
 9   date     184 non-null    str  
 10  notes    102 non-null    str  
dtypes: int64(3), str(8)
memory usage: 15.9 KB
dataframe descirebtion: 


,count,mean,std,min,25%,50%,75%,max
year,184.0,1984.260870,27.962844,1930.0,1962.0,1990.0,2010.0,2022.0
score1,184.0,2.358696,1.746700,0.0,1.0,2.0,3.0,9.0
score2,184.0,1.152174,1.436714,0.0,0.0,1.0,2.0,10.0


,count,unique,top,freq
stage,184,8,Group Stage,51
team1,184,39,Brazil,31
team2,184,44,England,11
venue,184,90,Estadio Centenario,8
city,184,86,Montevideo,8
country,184,19,France,17
date,184,160,1930-07-13,2
notes,102,96,AET,5


In [ ]:
dm = df_matches.copy()       
dm["manually_corrected"] = False     

In [ ]:
cov = dm.groupby("year").size().rename("rows").to_frame().join(df_editions.set_index("year")["matches"])
cov["coverage"] = (cov["rows"] / cov["matches"]).round(2)
print(cov, "\nTotal coverage:", round(len(dm) / df_editions["matches"].sum(), 2))
# we have only 20% of the actual world cup matches :(

      rows  matches  coverage
year                         
1930     8       18      0.44
1934     7       17      0.41
1938     7       18      0.39
1950     5       22      0.23
1954     7       26      0.27
1958     7       35      0.20
1962     7       32      0.22
1966     8       32      0.25
1970     8       32      0.25
1974     6       38      0.16
1978     5       38      0.13
1982     7       52      0.13
1986     8       52      0.15
1990     9       52      0.17
1994     8       52      0.15
1998    10       64      0.16
2002     9       64      0.14
2006    10       64      0.16
2010    11       64      0.17
2014    11       64      0.17
2018    12       64      0.19
2022    14       64      0.22 
Total coverage: 0.19


In [ ]:
FIXES = [  # (year, stage, team1, team2, {column: new value})
    (1970, "Group Stage",   "West Germany", "Bulgaria", {"score1": 5, "score2": 2, "date": "1970-06-03"}),
    (1930, "Group Stage",   "Argentina",    "USA",      {"team2": "Mexico", "date": "1930-07-19"}),
    (1982, "Quarter-final", "West Germany", "France",   {"stage": "Semi-final", "score1": 3, "score2": 3}),
    (1982, "Group Stage",   "Brazil",       "Argentina",{"stage": "Second Group Stage", "date": "1982-07-02"}),
    (1978, "Group Stage",   "Argentina",    "Peru",     {"stage": "Second Group Stage"}),
    (2014, "Quarter-final", "Argentina",    "Belgium",  {"score1": 1, "score2": 0}),
]

for year, stage, t1, t2, new in FIXES:
    hit = (dm["year"].eq(year) & dm["stage"].eq(stage)
        & dm["team1"].eq(t1) & dm["team2"].eq(t2))
    assert hit.sum() == 1, (year, t1, t2)      # must match exactly one row, else stop
    for col, val in new.items():
        dm.loc[hit, col] = val
    dm.loc[hit, "manually_corrected"] = True

In [38]:
for c in ("team1", "team2", "country"):
    dm[f"{c}_canon"] = dm[c].replace(CANON)
dm["city"] = dm["city"].replace({"Vina del Mar": "Viña del Mar", "Stanford": "Palo Alto"})

In [39]:
dm["date"] = pd.to_datetime(dm["date"], format="%Y-%m-%d")
assert (dm["date"].dt.year == dm["year"]).all()
assert all(pd.api.types.is_integer_dtype(dm[c]) for c in ("score1", "score2"))

In [40]:
dm["stage_clean"] = dm["stage"].replace({"Final Round": "Second Group Stage",
                                            "Semi-final Round": "Second Group Stage"})

STAGE_TYPE = {"Group Stage": "Group", "Second Group Stage": "Second group",
                "Round of 16": "Knockout", "Quarter-final": "Knockout",
                "Semi-final": "Knockout", "3rd Place": "Knockout", "Final": "Knockout"}
dm["stage_type"] = dm["stage_clean"].map(STAGE_TYPE)
assert dm["stage_type"].notna().all()

In [41]:
dm["total_goals"] = dm["score1"] + dm["score2"]
dm["went_aet"] = dm["notes"].str.contains("AET|pens", na=False)   # a shootout implies extra time
dm["host_playing"] = ((dm["team1_canon"] == dm["country_canon"])
                        | (dm["team2_canon"] == dm["country_canon"]))

def pens_winner(r):
    n = r["notes"]
    if not isinstance(n, str) or "pens" not in n:
        return None
    for t in (r["team1_canon"], r["team2_canon"]):   # notes say "Germany win", so compare canonical names
        if f"{t} win" in n:
            return t
    return None

dm["pens_winner"] = dm.apply(pens_winner, axis=1)
dm["decided_on_pens"] = dm["pens_winner"].notna()
assert not (dm["notes"].str.contains("pens", na=False) & ~dm["decided_on_pens"]).any()  # none unresolved

by_score = dm["team1_canon"].where(dm["score1"] > dm["score2"],
            dm["team2_canon"].where(dm["score2"] > dm["score1"]))
dm["winner"] = by_score.fillna(dm["pens_winner"])        # NaN = draw or replay


In [42]:
dm["pair_key"] = [" | ".join(sorted(p)) for p in zip(dm["team1_canon"], dm["team2_canon"])]
assert not dm.duplicated().any()
assert not dm.duplicated(["date", "pair_key"]).any()     # same two teams on the same day
assert dm["total_goals"].max() <= 12                     # the all-time record is 12 goals

fin = dm[dm["stage_clean"].eq("Final")].set_index("year")["date"]
assert fin.index.is_unique and len(fin) == 21            # 1950 had no final
assert (fin == dm.groupby("year")["date"].max().loc[fin.index]).all()   # the final is the last match


In [43]:
print(dm.loc[dm["winner"].isna() & dm["stage_type"].eq("Knockout"),
                ["year", "stage", "team1", "team2", "notes"]])   # expect only the 1934 and 1938 replays
vc = dm.groupby(["year", "venue"])["city"].nunique()
print(vc[vc > 1])  

    year          stage   team1           team2         notes
10  1934  Quarter-final   Italy           Spain  AET - replay
17  1938  Quarter-final  Brazil  Czechoslovakia        Replay
year  venue              
1934  PNF Stadium            2
      Stadio Bentegodi       2
1938  Stade de la Meinau     2
1950  Estádio do Maracanã    2
1954  Hardturm               2
      Wankdorf               2
1962  Estadio El Teniente    2
1974  Olympiastadion         3
2006  Allianz Arena          2
      Olympiastadion         2
Name: city, dtype: int64


In [44]:
dm.drop(columns="pair_key").to_csv("wc_all_matches_clean.csv", index=False, date_format="%Y-%m-%d")

### Fifth, the Scorers Dataset

**Data Frame Info**

In [45]:
print(f"\nShape:\n{df_scorers.shape}")
print(f"\nColumns:\n{df_scorers.columns.tolist()}")
print(f"\nData types:\n{df_scorers.dtypes}")
print("\nAll rows:")
display(df_scorers)
print("\nInfo:")
df_scorers.info()
print("dataframe descirebtion: ")
display(df_scorers.describe().T)
df_scorers.describe(include="str").T


Shape:
(22, 11)

Columns:
['edition', 'year', 'player', 'country', 'position', 'goals', 'assists', 'penalties', 'matches_played', 'host', 'team_result']

Data types:
edition           int64
year              int64
player              str
country             str
position            str
goals             int64
assists           int64
penalties         int64
matches_played    int64
host                str
team_result         str
dtype: object

All rows:


,edition,year,player,country,position,goals,assists,penalties,matches_played,host,team_result
0,1,1930,Guillermo Stábile,Argentina,FW,8,0,0,4,Uruguay,Champions
1,2,1934,Oldřich Nejedlý,Czechoslovakia,FW,5,0,0,4,Italy,Semi-final
2,3,1938,Leônidas,Brazil,FW,7,0,0,5,France,Third place
3,4,1950,Ademir,Brazil,FW,8,0,0,6,Brazil,Runner-up
4,5,1954,Sándor Kocsis,Hungary,FW,11,0,0,5,Switzerland,Runner-up
5,6,1958,Just Fontaine,France,FW,13,0,0,6,Sweden,Third place
6,7,1962,Multiple Winners,Multiple,FW,4,0,0,6,Chile,Varies
7,8,1966,Eusébio,Portugal,FW,9,0,0,6,England,Third place
8,9,1970,Gerd Müller,West Germany,FW,10,0,0,6,Mexico,Third place
9,10,1974,Grzegorz Lato,Poland,FW,7,0,0,7,West Germany,Third place



Info:
<class 'pandas.DataFrame'>
RangeIndex: 22 entries, 0 to 21
Data columns (total 11 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   edition         22 non-null     int64
 1   year            22 non-null     int64
 2   player          22 non-null     str  
 3   country         22 non-null     str  
 4   position        22 non-null     str  
 5   goals           22 non-null     int64
 6   assists         22 non-null     int64
 7   penalties       22 non-null     int64
 8   matches_played  22 non-null     int64
 9   host            22 non-null     str  
 10  team_result     22 non-null     str  
dtypes: int64(6), str(5)
memory usage: 2.0 KB
dataframe descirebtion: 


,count,mean,std,min,25%,50%,75%,max
edition,22.0,11.500000,6.493587,1.0,6.25,11.5,16.75,22.0
year,22.0,1978.909091,27.738419,1930.0,1959.00,1980.0,2001.00,2022.0
goals,22.0,7.090909,2.158242,4.0,6.00,6.0,8.00,13.0
assists,22.0,0.318182,0.838727,0.0,0.00,0.0,0.00,3.0
penalties,22.0,0.000000,0.000000,0.0,0.00,0.0,0.00,0.0
matches_played,22.0,5.954545,0.950051,4.0,5.25,6.0,7.00,7.0


,count,unique,top,freq
player,22,22,Guillermo Stábile,1
country,22,15,Brazil,3
position,22,1,FW,22
host,22,18,Italy,2
team_result,22,7,Third place,9


In [46]:
ds = df_scorers.copy()
ed = df_editions.set_index("year") # for refrence                   
ds["manually_corrected"] = False
ds["country_canon"] = ds["country"].replace(CANON)

In [ ]:
ds["player_all"] = ds["year"].map(ed["top_scorer"])
ds["n_co_leaders"] = ds["year"].map(ed["top_scorer"].str.count("/") + 1)
ds["is_tie"] = ds["n_co_leaders"] > 1 # 1962, 1994, and 2010

,edition,year,player,country,position,goals,assists,penalties,matches_played,host,team_result,manually_corrected,country_canon,player_all,n_co_leaders,is_tie
6,7,1962,Multiple Winners,Multiple,FW,4,0,0,6,Chile,Varies,False,Multiple,Florian Albert / Garrincha / Valentin Ivanov /...,6,True
14,15,1994,Hristo Stoichkov,Bulgaria,FW,6,0,0,6,United States,Semi-final,False,Bulgaria,Hristo Stoichkov / Oleg Salenko,2,True
18,19,2010,Thomas Müller,Germany,FW,5,3,0,6,South Africa,Third place,False,Germany,Thomas Müller / Wesley Sneijder / David Villa ...,4,True


In [48]:
ph = ds["player"].eq("Multiple Winners")
ds["matches_played"] = ds["matches_played"].astype("Int64").mask(ph)   # nullable int: can hold NA
ds["country_canon"] = ds["country_canon"].mask(ph)
ds["team_result"] = ds["team_result"].mask(ph)

In [49]:
ds["assists"] = ds["assists"].astype("Int64").mask(ds["year"] < 2010)


In [ ]:
placing = {}
for col, label in [("champion", "Champions"), ("runner_up", "Runner-up"),
                    ("third_place", "Third place"), ("fourth_place", "Fourth place")]:
    for y, team in zip(df_editions["year"], df_editions[col].replace(CANON)):
        placing[(y, team)] = label
derived = pd.Series([placing.get((y, c)) for y, c in zip(ds["year"], ds["country_canon"])],
                    index=ds.index)
bad = derived.notna() & (derived != ds["team_result"])
print(ds.loc[bad, ["year", "player", "country", "team_result"]].assign(derived=derived[bad]))  # 1930, 1934, 1994
ds.loc[bad, "team_result"] = derived[bad]
ds.loc[bad, "manually_corrected"] = True

    year             player         country team_result       derived
0   1930  Guillermo Stábile       Argentina   Champions     Runner-up
1   1934    Oldřich Nejedlý  Czechoslovakia  Semi-final     Runner-up
14  1994   Hristo Stoichkov        Bulgaria  Semi-final  Fourth place


In [51]:
chk = ds.set_index("year")
assert (chk["goals"] == ed["top_scorer_goals"].reindex(chk.index)).all()
norm = lambda s: s.str.replace(r"\s*/\s*", " / ", regex=True)
assert (norm(chk["host"]) == norm(ed["host"]).reindex(chk.index)).all()
ds = ds.drop(columns=["penalties", "position", "host"])
assert ds["year"].is_unique

In [52]:
print(ds["country_canon"].value_counts().head(4))
print(ds.loc[~ds["is_tie"], "country_canon"].value_counts().head(4))

country_canon
Brazil       3
Germany      3
Argentina    2
France       2
Name: count, dtype: int64
country_canon
Brazil       3
Argentina    2
France       2
Germany      2
Name: count, dtype: int64


In [53]:
ds.to_csv("wc_top_scorers_clean.csv", index=False)